从给定的CSV文件读取数据并生成Spark数据框架

In [2]:
import os
os.environ["JAVA_HOME"] = "C:\\Program Files\\JDK1.8"
os.environ["SPARK_HOME"] = "C:\\spark-3.4.0-bin-without-hadoop"

In [3]:
from pyspark.sql import SparkSession

# 使用现代Spark初始化方式
spark = SparkSession.builder \
    .appName("MyApp") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

输出数据的前10行

In [6]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("file:///D:/code/python/BigDataDevelopment/myCode/Homework4_datasets/epi_r.csv")

df.limit(10).toPandas()

,title,rating,calories,protein,fat,sodium,#cakeweek,#wasteless,22-minute meals,3-ingredient recipes,...,yellow squash,yogurt,yonkers,yuca,zucchini,cookbooks,leftovers,snack,snack week,turkey
0,"Lentil, Apple, and Turkey Wrap",2.5,426.0,30.0,7.0,559.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,Boudin Blanc Terrine with Red Onion Confit,4.375,403.0,18.0,23.0,1439.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Potato and Fennel Soup Hodge,3.75,165.0,6.0,7.0,165.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Mahi-Mahi in Tomato Olive Sauce,5.0,None,NaN,NaN,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Spinach Noodle Casserole,3.125,547.0,20.0,32.0,452.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,The Best Blts,4.375,948.0,19.0,79.0,1042.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,Ham and Spring Vegetable Salad with Shallot Vi...,4.375,None,NaN,NaN,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,Spicy-Sweet Kumquats,3.75,None,NaN,NaN,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,Korean Marinated Beef,4.375,170.0,7.0,10.0,1272.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,Ham Persillade with Mustard Potato Salad and M...,3.75,602.0,23.0,41.0,1696.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


打印输出数据模式Schema及其变量名列表

In [7]:
df.printSchema()
print(df.columns)

root
 |-- title: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- calories: string (nullable = true)
 |-- protein: double (nullable = true)
 |-- fat: double (nullable = true)
 |-- sodium: double (nullable = true)
 |-- #cakeweek: double (nullable = true)
 |-- #wasteless: double (nullable = true)
 |-- 22-minute meals: double (nullable = true)
 |-- 3-ingredient recipes: double (nullable = true)
 |-- 30 days of groceries: double (nullable = true)
 |-- advance prep required: double (nullable = true)
 |-- alabama: double (nullable = true)
 |-- alaska: double (nullable = true)
 |-- alcoholic: double (nullable = true)
 |-- almond: double (nullable = true)
 |-- amaretto: double (nullable = true)
 |-- anchovy: double (nullable = true)
 |-- anise: double (nullable = true)
 |-- anniversary: double (nullable = true)
 |-- anthony bourdain: double (nullable = true)
 |-- aperitif: double (nullable = true)
 |-- appetizer: double (nullable = true)
 |-- apple: double (nullable = true)


打印输出数据框架的行数和列数

In [33]:
print("行数:", df.count())
print("列数:", len(df.columns))

行数: 20057
列数: 680


打印输出整个数据框架的汇总统计量，并任选两个单独的数值列输出其汇总统计量

In [ ]:
from pyspark.sql.types import NumericType

# 重命名不规范列名
new_column_names = []

for old_name in df.columns:
    new_name = old_name

    new_name = new_name.replace(',', '')
    new_name = new_name.replace('.', '')
    new_name = new_name.replace(' ', '_')
    new_name = new_name.replace('-', '_')
    new_name = new_name.replace('#', '')
    new_name = new_name.replace('&', 'and')
    new_name = new_name.replace("'", '')
    new_name = new_name.replace('(', '')
    new_name = new_name.replace(')', '')
    new_name = new_name.replace('+', '_')
    new_name = new_name.replace('/', '_or_')

    while '__' in new_name:
        new_name = new_name.replace('__', '_')

    new_name = new_name.strip('_')

    if new_name and new_name[0].isdigit():
        new_name = 'col_' + new_name

    if not new_name:
        new_name = 'unknown_column'

    new_column_names.append(new_name)

df_safe = df.toDF(*new_column_names)

numeric_cols = [col.name for col in df_safe.schema.fields if isinstance(col.dataType, NumericType)]
df_safe.describe(numeric_cols).show(20, truncate=False)

重命名完成！原始列数: 680, 新列数: 680


In [ ]:
if len(numeric_cols) >= 2:
    selected_cols = numeric_cols[:2]
    df_safe.describe(selected_cols).show(truncate=False)